# Resume Screening & Job Recommendation System

**NLP + TF-IDF + Cosine Similarity**

Resume → Cleaning → TF-IDF → Similarity → Screening → Job Recommendation → Skill Gap

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
resumes=pd.read_csv("data/resumes.csv")
jobs=pd.read_csv("data/jobs.csv")
display(resumes.head())
display(jobs[["job_id","job_title","company"]])


,resume_id,candidate_name,resume_text,experience_years
0,R001,Candidate 1,Python developer with pandas numpy scikit-lear...,2
1,R002,Candidate 2,Frontend developer skilled in HTML CSS JavaScr...,2
2,R003,Candidate 3,Data science student with Python pandas numpy ...,1
3,R004,Candidate 4,Backend developer using Python Flask Django SQ...,3
4,R005,Candidate 5,Java developer experienced in Java Spring Boot...,3


,job_id,job_title,company
0,J001,Junior Data Scientist,DataTech
1,J002,Machine Learning Engineer,AI Labs
2,J003,Frontend Developer,WebWorks
3,J004,Python Backend Developer,CloudApps
4,J005,NLP Engineer Intern,LanguageAI


In [4]:
def clean_text(text):
    text=str(text).lower()
    text=re.sub(r"[^a-z0-9+#.\s]"," ",text)
    return re.sub(r"\s+"," ",text).strip()

resumes["clean_resume"]=resumes["resume_text"].apply(clean_text)
jobs["clean_job"]=jobs["job_description"].apply(clean_text)


In [5]:
all_text=pd.concat([resumes["clean_resume"],jobs["clean_job"]],ignore_index=True)
vectorizer=TfidfVectorizer(stop_words="english",ngram_range=(1,2))
tfidf=vectorizer.fit_transform(all_text)
resume_vectors=tfidf[:len(resumes)]
job_vectors=tfidf[len(resumes):]
similarity=cosine_similarity(resume_vectors,job_vectors)
print("TF-IDF shape:",tfidf.shape)


TF-IDF shape: (11, 164)


In [7]:
recommendations=[]
for i,r in resumes.iterrows():
    top=np.argsort(similarity[i])[::-1][:3]
    for rank,j in enumerate(top,1):
        recommendations.append({
            "resume_id":r.resume_id,
            "candidate":r.candidate_name,
            "rank":rank,
            "job_id":jobs.iloc[j].job_id,
            "job_title":jobs.iloc[j].job_title,
            "company":jobs.iloc[j].company,
            "match_score":round(similarity[i,j]*100,2)
        })
recommendations=pd.DataFrame(recommendations)
display(recommendations)


,resume_id,candidate,rank,job_id,job_title,company,match_score
0,R001,Candidate 1,1,J001,Junior Data Scientist,DataTech,31.73
1,R001,Candidate 1,2,J002,Machine Learning Engineer,AI Labs,15.37
2,R001,Candidate 1,3,J005,NLP Engineer Intern,LanguageAI,14.21
3,R002,Candidate 2,1,J003,Frontend Developer,WebWorks,71.60
4,R002,Candidate 2,2,J004,Python Backend Developer,CloudApps,10.61
5,R002,Candidate 2,3,J005,NLP Engineer Intern,LanguageAI,0.00
6,R003,Candidate 3,1,J001,Junior Data Scientist,DataTech,41.53
7,R003,Candidate 3,2,J002,Machine Learning Engineer,AI Labs,24.09
8,R003,Candidate 3,3,J005,NLP Engineer Intern,LanguageAI,14.88
9,R004,Candidate 4,1,J004,Python Backend Developer,CloudApps,77.81


In [8]:
selected_job="J001"
j=jobs.index[jobs.job_id==selected_job][0]
screen=resumes[["resume_id","candidate_name","experience_years"]].copy()
screen["match_score"]=(similarity[:,j]*100).round(2)
screen["status"]=np.where(screen.match_score>=45,"Shortlisted","Needs Review")
screen=screen.sort_values("match_score",ascending=False)
display(screen)


,resume_id,candidate_name,experience_years,match_score,status
2,R003,Candidate 3,1,41.53,Needs Review
0,R001,Candidate 1,2,31.73,Needs Review
5,R006,Candidate 6,1,18.09,Needs Review
3,R004,Candidate 4,3,3.37,Needs Review
1,R002,Candidate 2,2,0.00,Needs Review
4,R005,Candidate 5,3,0.00,Needs Review


In [9]:
skills=["python","pandas","numpy","sql","machine learning","scikit-learn",
"statistics","data analysis","matplotlib","seaborn","xgboost","tensorflow",
"nlp","html","css","javascript","react","git","rest api","flask","django",
"postgresql","docker","linux","text classification"]

def extract_skills(text):
    text=clean_text(text)
    return sorted({s for s in skills if s in text})

resumes["skills"]=resumes.resume_text.apply(extract_skills)
display(resumes[["candidate_name","skills"]])


,candidate_name,skills
0,Candidate 1,"[data analysis, flask, machine learning, matpl..."
1,Candidate 2,"[css, git, html, javascript, react, rest api]"
2,Candidate 3,"[machine learning, matplotlib, numpy, pandas, ..."
3,Candidate 4,"[django, docker, flask, git, linux, postgresql..."
4,Candidate 5,"[git, rest api, sql]"
5,Candidate 6,"[machine learning, nlp, numpy, pandas, python,..."


In [10]:
resume_id="R003"
candidate=resumes[resumes.resume_id==resume_id].iloc[0]
job=jobs.iloc[j]
candidate_skills=set(candidate.skills)
required=set(x.strip().lower() for x in job.required_skills.split(","))
matched=sorted(candidate_skills & required)
missing=sorted(required-candidate_skills)
print("Candidate:",candidate.candidate_name)
print("Job:",job.job_title)
print("Matched skills:",matched)
print("Missing skills:",missing)
print("Skill match:",round(len(matched)/len(required)*100,2),"%")


Candidate: Candidate 3
Job: Junior Data Scientist
Matched skills: ['machine learning', 'matplotlib', 'numpy', 'pandas', 'python', 'seaborn', 'sql', 'statistics']
Missing skills: ['data analysis', 'scikit-learn']
Skill match: 80.0 %


In [11]:
def recommend_jobs(resume_id,top_n=3):
    idx=resumes.index[resumes.resume_id==resume_id][0]
    out=jobs[["job_id","job_title","company"]].copy()
    out["match_score"]=(similarity[idx]*100).round(2)
    return out.sort_values("match_score",ascending=False).head(top_n)

display(recommend_jobs("R003",3))


,job_id,job_title,company,match_score
0,J001,Junior Data Scientist,DataTech,41.53
1,J002,Machine Learning Engineer,AI Labs,24.09
4,J005,NLP Engineer Intern,LanguageAI,14.88


In [12]:
recommendations.to_csv("job_recommendations.csv",index=False)
screen.to_csv("resume_screening_results.csv",index=False)
print("Results saved.")


Results saved.


## Conclusion
यह academic prototype TF-IDF और Cosine Similarity से resumes को jobs से match करता है, candidates को rank करता है और basic skill-gap analysis देता है। Real recruitment system में human review, privacy और fairness checks भी जरूरी हैं।
